# Test Azure OpenAI endpoints

Verifies both deployments can be reached, returns valid responses, and
reports latency. Useful to:
- Confirm credentials work after a key rotation
- Compare latency between endpoints
- Pick which one to use for production runs

**Security note:** This notebook expects credentials in `.env` (or env
variables) - never paste keys directly into cells. If your `.env` doesn't
exist, see the example file path printed below.


## 0. Config - load credentials from .env

In [ ]:
import os
from pathlib import Path

# Load .env if python-dotenv is installed (recommended)
try:
    from dotenv import load_dotenv
    env_path = Path.cwd() / '.env'
    if env_path.exists():
        load_dotenv(env_path)
        print(f'Loaded credentials from {env_path}')
    else:
        print(f'No .env file at {env_path}')
        print(f'Create one with:')
        print(f'  ENDPOINT_A_URL=https://joshu-mebu0lvp-swedencentral.cognitiveservices.azure.com/')
        print(f'  ENDPOINT_A_KEY=<key-from-azure-portal>')
        print(f'  ENDPOINT_A_DEPLOYMENT=gpt-5-mini_ng')
        print(f'  ENDPOINT_B_URL=https://comissio-poc.openai.azure.com/')
        print(f'  ENDPOINT_B_KEY=<key-from-azure-portal>')
        print(f'  ENDPOINT_B_DEPLOYMENT=gpt-5-mini_ng')
except ImportError:
    print('python-dotenv not installed - reading from os.environ directly')
    print('pip install python-dotenv  # if you want to use .env files')

API_VERSION = '2024-12-01-preview'

# Build endpoint configs
endpoints = {
    'A_swedencentral': {
        'url':        os.getenv('ENDPOINT_A_URL'),
        'key':        os.getenv('ENDPOINT_A_KEY'),
        'deployment': os.getenv('ENDPOINT_A_DEPLOYMENT', 'gpt-5-mini_ng'),
    },
    'B_comissio_poc': {
        'url':        os.getenv('ENDPOINT_B_URL'),
        'key':        os.getenv('ENDPOINT_B_KEY'),
        'deployment': os.getenv('ENDPOINT_B_DEPLOYMENT', 'gpt-5-mini_ng'),
    },
}

print('Configured endpoints:')
for name, cfg in endpoints.items():
    has_url = bool(cfg['url'])
    has_key = bool(cfg['key'])
    print(f'  {name:<20} url={has_url}  key={has_key}  deployment={cfg["deployment"]}')


## 1. Helper - call any endpoint with a test prompt

Defensive HTTP with status code and body shown on any non-200.

In [ ]:
import json
import time
import httpx


def call_endpoint(cfg, prompt, max_tokens=200, timeout=30):
    '''Make one chat completion call and return (success, response_text, latency_s, diagnostic).'''
    if not cfg['url'] or not cfg['key']:
        return False, None, 0.0, 'missing url or key in config'

    url = (
        f"{cfg['url'].rstrip('/')}/openai/deployments/{cfg['deployment']}"
        f"/chat/completions?api-version={API_VERSION}"
    )
    headers = {'api-key': cfg['key'], 'Content-Type': 'application/json'}
    body = {
        'messages': [
            {'role': 'system', 'content': 'You are a helpful assistant. Respond concisely.'},
            {'role': 'user',   'content': prompt},
        ],
        'max_completion_tokens': max_tokens,
        # NB: gpt-5 family doesn't accept explicit temperature - omit it.
    }

    t0 = time.monotonic()
    try:
        resp = httpx.post(url, headers=headers, json=body, timeout=timeout)
    except httpx.RequestError as e:
        return False, None, time.monotonic() - t0, f'network error: {e}'
    latency = time.monotonic() - t0

    if resp.status_code != 200:
        body_preview = (resp.text or '')[:400]
        return False, None, latency, f'HTTP {resp.status_code}: {body_preview}'

    try:
        data = resp.json()
    except json.JSONDecodeError:
        return False, None, latency, f'non-JSON response: {resp.text[:400]}'

    try:
        content = data['choices'][0]['message']['content']
        finish  = data['choices'][0].get('finish_reason', 'unknown')
        usage   = data.get('usage', {})
        diag = f"finish={finish} | tokens p/c/t = {usage.get('prompt_tokens')}/{usage.get('completion_tokens')}/{usage.get('total_tokens')}"
        return True, content, latency, diag
    except (KeyError, IndexError) as e:
        return False, None, latency, f'unexpected response shape: {e} | {str(data)[:400]}'


print('Helper ready')


## 2. Smoke test - simple prompt on each endpoint

In [ ]:
TEST_PROMPT = 'Say the word "working" and nothing else.'

print(f'Prompt: {TEST_PROMPT}\n')
print('=' * 70)

results = {}
for name, cfg in endpoints.items():
    print(f'\nTesting {name} ...')
    print(f'  URL:        {cfg["url"]}')
    print(f'  Deployment: {cfg["deployment"]}')
    # gpt-5 family uses completion tokens for reasoning - need generous budget
    ok, content, latency, diag = call_endpoint(cfg, TEST_PROMPT, max_tokens=500)
    results[name] = {'ok': ok, 'content': content, 'latency': latency, 'diag': diag}
    if ok:
        print(f'  [OK]    {latency:.2f}s  |  {diag}')
        print(f'  Response: {content!r}')
    else:
        print(f'  [FAIL]  {latency:.2f}s')
        print(f'  Error:  {diag}')


## 3. Realistic prompt test - sized like the v6 pipeline

The smoke test above uses a tiny prompt. This cell sends something
closer to what the actual pipeline sends per benefit (~2700 chars of
system + few-shot + ~500 chars of user message). Tells you what
real-world latency to expect.

In [ ]:
# Approximate the v6 system prompt size
REALISTIC_SYSTEM = '''You are a Medicare Advantage benefit description formatter.

You receive PBP input rows for ONE specific benefit and produce TWO text fields:
  benefitdesc      - HTML-formatted description for display to a consumer
  tinyDescription  - 5-15 character plain-text summary

NEVER output "Not Applicable". If no cost data is available, output "See Brochure".

FORMATTING RULES:
- Dollar amounts and percentages in <b> tags: <b>$25</b>, <b>20%</b>
- Copay: <b>$X</b> copay
- Coinsurance: <b>X%</b> of the cost
- Strip .00 from dollars: $7.00 -> $7
- Not covered -> "Not Covered"

OUTPUT: One JSON object only:
{"benefitdesc": "...", "tinyDescription": "..."}
'''

REALISTIC_USER = '''Benefit: 900 - Doctor Office Visits Primary
Plan Type: HMO
Coverage: InNetwork  Service: N/A

Input rows:
  [Primary Care Physician Services (7a) - Medicare] Is there a copayment? = Yes
  [Primary Care Physician Services (7a) - Medicare] Copayment amount = $7.00
  [Primary Care Physician Services (7a) - Medicare] Is there a coinsurance? = No

Return ONLY a JSON object with two fields.
'''

print(f'System prompt: {len(REALISTIC_SYSTEM)} chars')
print(f'User message: {len(REALISTIC_USER)} chars\n')


def call_endpoint_realistic(cfg, timeout=60):
    '''Send the realistic prompt - same shape as v6 pipeline.'''
    if not cfg['url'] or not cfg['key']:
        return False, None, 0.0, 'missing url or key'
    url = (
        f"{cfg['url'].rstrip('/')}/openai/deployments/{cfg['deployment']}"
        f"/chat/completions?api-version={API_VERSION}"
    )
    headers = {'api-key': cfg['key'], 'Content-Type': 'application/json'}
    body = {
        'messages': [
            {'role': 'system', 'content': REALISTIC_SYSTEM},
            {'role': 'user',   'content': REALISTIC_USER},
        ],
        'max_completion_tokens': 200,
    }
    t0 = time.monotonic()
    try:
        resp = httpx.post(url, headers=headers, json=body, timeout=timeout)
    except httpx.RequestError as e:
        return False, None, time.monotonic() - t0, f'network: {e}'
    latency = time.monotonic() - t0
    if resp.status_code != 200:
        return False, None, latency, f'HTTP {resp.status_code}: {resp.text[:300]}'
    data = resp.json()
    content = data['choices'][0]['message']['content']
    usage = data.get('usage', {})
    diag = f"tokens p/c/t = {usage.get('prompt_tokens')}/{usage.get('completion_tokens')}/{usage.get('total_tokens')}"
    return True, content, latency, diag

print('=' * 70)
realistic_results = {}
for name, cfg in endpoints.items():
    print(f'\nTesting {name} with realistic prompt ...')
    ok, content, latency, diag = call_endpoint_realistic(cfg)
    realistic_results[name] = {'ok': ok, 'content': content, 'latency': latency, 'diag': diag}
    if ok:
        print(f'  [OK]   {latency:.2f}s  |  {diag}')
        print(f'  Response: {content[:200]}')
    else:
        print(f'  [FAIL] {latency:.2f}s  |  {diag}')


## 4. Burst test - 5 quick calls to detect throttling

Hits each endpoint 5 times in quick succession. If you see 429 errors,
that's your TPM cap - and it explains the slowness in v6 runs.

In [ ]:
def burst_test(name, cfg, n_calls=5):
    print(f'\n--- Burst test: {name} ---')
    results = []
    t_start = time.monotonic()
    for i in range(n_calls):
        ok, content, latency, diag = call_endpoint(cfg, 'Reply with one word.', max_tokens=300, timeout=30)
        results.append({'ok': ok, 'latency': latency, 'diag': diag})
        flag = 'OK' if ok else 'FAIL'
        print(f'  [{i+1}/{n_calls}] [{flag}] {latency:.2f}s  {diag if not ok else ""}')
    total = time.monotonic() - t_start
    n_ok = sum(1 for r in results if r['ok'])
    print(f'  Summary: {n_ok}/{n_calls} succeeded in {total:.1f}s  '
          f'(avg {total/n_calls:.2f}s/call)')
    has_429 = any('429' in (r['diag'] or '') for r in results)
    if has_429:
        print(f'  WARNING: 429 throttling detected - TPM cap is being hit')
    return results

for name, cfg in endpoints.items():
    if cfg['url'] and cfg['key']:
        burst_test(name, cfg, n_calls=5)


## 5. Summary - which endpoint to use

In [ ]:
print('=' * 70)
print('SUMMARY')
print('=' * 70)

for name in endpoints:
    smoke = results.get(name, {})
    real  = realistic_results.get(name, {})
    print(f'\n{name}:')
    print(f'  Smoke test:     {"OK" if smoke.get("ok") else "FAIL"}  ({smoke.get("latency", 0):.2f}s)')
    print(f'  Realistic:      {"OK" if real.get("ok") else "FAIL"}  ({real.get("latency", 0):.2f}s)')
    if not smoke.get('ok'):
        print(f'    Error: {smoke.get("diag")}')

# Recommendation
working = [n for n, r in results.items() if r.get('ok') and realistic_results.get(n, {}).get('ok')]
print(f'\nWorking endpoints: {working}')

if len(working) == 2:
    a_lat = realistic_results['A_swedencentral']['latency']
    b_lat = realistic_results['B_comissio_poc']['latency']
    faster = 'A_swedencentral' if a_lat < b_lat else 'B_comissio_poc'
    print(f'\nFaster endpoint on realistic prompts: {faster}')
    print(f'  A: {a_lat:.2f}s   B: {b_lat:.2f}s   diff: {abs(a_lat-b_lat):.2f}s')
    print(f'\nTo use {faster} in the v6 pipeline, set in .env:')
    cfg = endpoints[faster]
    print(f'  AZURE_OPENAI_ENDPOINT={cfg["url"]}')
    print(f'  AZURE_OPENAI_API_KEY=<key for {faster}>')
    print(f'  AZURE_OPENAI_DEPLOYMENT={cfg["deployment"]}')
    print(f'  AZURE_OPENAI_API_VERSION={API_VERSION}')
elif len(working) == 1:
    print(f'\nUse {working[0]} - the other endpoint is not functional')
else:
    print(f'\nNeither endpoint worked - check credentials and network')
